In [ ]:
!pip install pandas pyarrow nltk spacy transformers torch datasets scikit-learn
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 122.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# =============================================
# FAKE REVIEW DETECTION - FULL COLAB VERSION
# =============================================

import pandas as pd
import numpy as np
import re
import nltk
import spacy
import torch

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

# ==========================
# CHECK DEVICE
# ==========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==========================
# DOWNLOAD NLTK
# ==========================
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))
nlp = spacy.load("en_core_web_sm")

# ==========================
# LOAD DATASET
# ==========================
print("Loading dataset...")
df = pd.read_parquet(
    "hf://datasets/theArijitDas/Fake-Reviews-Dataset/data/train-00000-of-00001.parquet"
)
print("Dataset shape:", df.shape)

# ==========================
# CLEAN TEXT
# ==========================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    doc = nlp(text)
    tokens = [
        token.lemma_
        for token in doc
        if token.text not in stop_words and not token.is_punct
    ]
    return " ".join(tokens)

print("Cleaning text...")
df["clean_text"] = df["text"].apply(clean_text)

# ==========================
# SPLIT DATA
# ==========================
train_df, test_df = train_test_split(
    df[["clean_text", "label"]],
    test_size=0.2,
    random_state=42
)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# ==========================
# TOKENIZER
# ==========================
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(
        example["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# ==========================
# LOAD MODEL
# ==========================
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)
model.to(device)

# ==========================
# METRICS FUNCTION
# ==========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions)
    recall = recall_score(labels, predictions)
    f1 = f1_score(labels, predictions)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

# ==========================
# TRAINING ARGUMENTS
# ==========================
training_args = TrainingArguments(
    output_dir="/content/results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="/content/logs",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    fp16=True
)

# ==========================
# TRAINER
# ==========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# ==========================
# TRAIN MODEL
# ==========================
print("Training model...")
trainer.train()

# ==========================
# FINAL EVALUATION
# ==========================
print("Evaluating...")
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

accuracy = accuracy_score(test_df["label"], preds)
precision = precision_score(test_df["label"], preds)
recall = recall_score(test_df["label"], preds)
f1 = f1_score(test_df["label"], preds)

print("\n===== FINAL METRICS =====")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("\nClassification Report:\n")
print(classification_report(test_df["label"], preds))

# ==========================
# CONFIDENCE SCORE
# ==========================
probs = torch.nn.functional.softmax(
    torch.tensor(predictions.predictions),
    dim=1
)

real_prob = probs[:, 0].numpy()
fake_prob = probs[:, 1].numpy()
confidence = np.max(
    np.vstack([real_prob, fake_prob]).T,
    axis=1
)

# ==========================
# EXPORT CSV TO DRIVE
# ==========================
results_df = pd.DataFrame({
    "text": test_df["clean_text"].values,
    "true_label": test_df["label"].values,
    "predicted_label": preds,
    "real_probability": real_prob,
    "fake_probability": fake_prob,
    "confidence_score": confidence
})

csv_path = "/content/drive/MyDrive/evaluation_results.csv"
results_df.to_csv(csv_path, index=False)

print("CSV saved to:", csv_path)

# ==========================
# SAVE MODEL TO DRIVE
# ==========================
model_path = "/content/drive/MyDrive/fake_review_model"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

print("Model saved to:", model_path)


Using device: cuda


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Loading dataset...
Dataset shape: (40526, 4)
Cleaning text...


Map:   0%|          | 0/32420 [00:00<?, ? examples/s]

Map:   0%|          | 0/8106 [00:00<?, ? examples/s]

...

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Evaluating...



===== FINAL METRICS =====
Accuracy : 0.9080927707870713
Precision: 0.8677120141342756
Recall   : 0.9641717791411043
F1 Score : 0.9134023015227246

Classification Report:

              precision    recall  f1-score   support

           0       0.96      0.85      0.90      4031
           1       0.87      0.96      0.91      4075

    accuracy                           0.91      8106
   macro avg       0.91      0.91      0.91      8106
weighted avg       0.91      0.91      0.91      8106

CSV saved to: /content/drive/MyDrive/evaluation_results.csv


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/drive/MyDrive/fake_review_model
